In [1]:
import json
import time
import pytz
import requests
from pathlib import Path
from datetime import datetime,timezone,timedelta
from scipy.optimize import curve_fit
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from bot_template import BaseBot, OrderBook, Order, OrderRequest, OrderResponse, Trade, Side, Product

EXCHANGE_URL  = "http://ec2-52-49-69-152.eu-west-1.compute.amazonaws.com/"   
USERNAME = ""  
PASSWORD = "" 

LONDON_TZ = pytz.timezone('Europe/London')

In [2]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

def get_weather(past_steps=96, forecast_steps=96):
    """伦敦15分钟天气预报, 返回一个包含以下列的DataFrame: 时间, 温度, 湿度."""
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "humidity": m["relative_humidity_2m"],
    })

In [3]:
def get_wx_spot():
    weather_df = get_weather()
    weather_df = weather_df[(weather_df["time"]<=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)))&(weather_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)))]
    weather_df["temperature"] = weather_df["temperature"]*9/5+32
    wx_spot = weather_df[weather_df["time"]==LONDON_TZ.localize(datetime(2026,3,1,12,0,0))]["temperature"] * weather_df[weather_df["time"]==LONDON_TZ.localize(datetime(2026,3,1,12,0,0))]["humidity"]
    return round(wx_spot.item())

def get_wx_sum():
    weather_df = get_weather()
    weather_df = weather_df[(weather_df["time"]<=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)))&(weather_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)))]
    weather_df["temperature"] = weather_df["temperature"]*9/5+32
    weather_df["temp_hum_product"] = weather_df["temperature"]*weather_df["humidity"]
    wx_sum = weather_df["temp_hum_product"].sum()/100
    return round(wx_sum.item())

In [4]:
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

def get_thames(limit=400):
    """获取威斯敏斯特地区泰晤士河的最新潮汐数据. 返回一个包含以下列的DataFrame: 时间, 水平"""
    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": limit},
    )
    resp.raise_for_status()
    items = resp.json().get("items", [])
    df = pd.DataFrame(items)[["dateTime", "value"]].rename(columns={"dateTime": "time", "value": "level"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/London")
    return df.sort_values("time").reset_index(drop=True)

In [5]:
def sin_function(x,a,b,c,d):
    """正弦拟合函数y=a+bsin(cx+d)"""
    return a+b*np.sin(c*x+d)

def fit_thames_data(thames_df):
    """用正弦函数y=a+bsin(cx+d)拟合水位数据, 返回拟合参数(a,b,c,d)"""
    start_time = thames_df["time"].min()
    hours_since_start = (thames_df["time"]-start_time).dt.total_seconds()/3600
    x_data = hours_since_start.values
    y_data = thames_df["level"].values
    initial_guess = [np.mean(y_data),np.std(y_data),2*np.pi/12,0] 
    popt, _ = curve_fit(sin_function,x_data,y_data,p0=initial_guess,maxfev=5000,)
    a,b,c,d = popt
    return a,b,c,d


In [6]:
def get_tide_spot():
    thames_df = get_thames()
    a,b,c,d = fit_thames_data(thames_df)
    predicted_level = sin_function((LONDON_TZ.localize(datetime(2026,3,1,12,0,0))-thames_df["time"].min()).total_seconds()/3600,a,b,c,d)
    tide_spot = abs(predicted_level)*1000
    return round(tide_spot.item())

def get_tide_swing():
    thames_df = get_thames()
    thames_df = thames_df[thames_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0))]
    a,b,c,d = fit_thames_data(thames_df)
    
    date_range = pd.date_range(start=max(thames_df['time'])+timedelta(minutes=15),end=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)),freq="15min",tz="Europe/London")
    thames_missing_df = pd.DataFrame({'time':date_range})
    hours_since_start = (thames_missing_df["time"]-thames_df["time"].min()).dt.total_seconds()/3600
    thames_missing_df["level"] = a+b*np.sin(c*hours_since_start.values+d)
    thames_df = pd.concat([thames_df,thames_missing_df], ignore_index=True).sort_values("time").drop_duplicates(subset=["time"], keep="last").reset_index(drop=True)

    thames_df["level_cm"] = round(thames_df["level"]*100)
    thames_df["diff_cm"] = thames_df["level_cm"].diff().abs().dropna().reset_index(drop=True)
    thames_df["put_payoff"] = np.maximum(0,20-thames_df["diff_cm"])
    thames_df["call_payoff"] = np.maximum(0,thames_df["diff_cm"]-25)
    thames_df["total_payoff"] = thames_df["put_payoff"]+thames_df["call_payoff"]

    return thames_df["total_payoff"].sum()


In [7]:
AERODATABOX_KEY = "3765cc9834msh09d931ddcb4dd6fp17befcjsn20f0b91eb55f" 
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
AIRPORT = "LHR"

def fetch_flights(airport=AIRPORT,offset_minutes=-360,duration_minutes=720,filters:dict|None=None):
    """按相对时间窗口(从现在开始的偏移量)获取航班信息. 该API抓取次数有限制所以说要保留历史文档"""
    params = f"?offsetMinutes={offset_minutes}&durationMinutes={duration_minutes}&direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

def fetch_flights_range(airport=AIRPORT,from_local="2026-02-28T12:00",to_local="2026-02-29T00:00",filters: dict|None = None):
    """根据明确的本地时间范围(最大跨度为12小时)获取航班信息. 该API抓取次数有限制所以说要保留历史文档"""
    params = "?direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}/{from_local}/{to_local}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

In [8]:
flights_data_1 = fetch_flights_range(from_local="2026-02-28T12:00",to_local="2026-03-01T00:00")

In [9]:
flights_data_2 = fetch_flights_range(from_local="2026-03-01T00:00",to_local="2026-03-01T12:00")

In [10]:
def get_lhr_count():
    global flights_data_1, flights_data_2    
    lhr_count = len(flights_data_1.get('arrivals',[]))+len(flights_data_1.get('departures',[]))+len(flights_data_2.get('arrivals',[]))+len(flights_data_2.get('departures',[]))
    return lhr_count

In [11]:
def flight_time(flight_record,flight_type):
    """记录航班的时间. 先用实际落地/起飞时间, 否则用计划时间"""
    time_fields = ["runwayTime","revisedTime","scheduledTime"]
    movement = flight_record["movement"]
    for field in time_fields:
        if field in movement and movement[field].get("utc"):
            flight_time = movement[field]["utc"]
            break
    flight_time = pytz.utc.localize(datetime.strptime(flight_time,"%Y-%m-%d %H:%MZ")).astimezone(LONDON_TZ)
    return round(flight_time)

def group_flights(flights_data,start_time,end_time):
    """按30分钟区间分组统计到达/出发航班数"""
    arrival_times = []
    for arr in flights_data.get("arrivals",[]):
        try:
            arr_time = parse_flight_time(arr,"arrival")
            if start_time<=arr_time<end_time:
                arrival_times.append(arr_time)
        except:
            pass
    departure_times = []
    for dep in flights_data.get("departures", []):
        try:
            dep_time = parse_flight_time(dep, "departure")
            if start_time<=dep_time<end_time: 
                departure_times.append(dep_time)
        except:
            pass
    return (len(arrival_times),len(departure_times))

def get_lhr_index():
    global flights_data_1, flights_data_2
    lhr_index = 0
    date_range = pd.date_range(start=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)),end=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)),freq="30min",tz="Europe/London")
    for index in range(24):
        arr,dep = group_flights(flights_data_1,date_range[index],date_range[index+1])
        lhr_index += ((arr-dep)/(arr+dep))*100 if arr+dep else 0
    for index in range(24):
        arr,dep = group_flights(flights_data_2,date_range[index+24],date_range[index+25])
        lhr_index += ((arr-dep)/(arr+dep))*100 if arr+dep else 0
    lhr_index = abs(lhr_index)
    return round(lhr_index)


In [12]:
def get_settlement_price(symbol):
    price_map = {"TIDE_SPOT": get_tide_spot(),"TIDE_SWING": get_tide_swing(),"WX_SUM": get_wx_sum(),"LHR_COUNT": get_lhr_count(),"LHR_INDEX": get_lhr_index(),"LON_ETF": get_lon_etf(),"LON_FLY": get_lon_fly()}
    return price_map.get(symbol, 10.0)

def get_lon_etf():
    return get_tide_spot()+get_wx_spot()+get_lhr_count()

def get_lon_fly():
    etf_price = get_lon_etf() 
    part1 = 2*max(6200-etf_price,0)
    part2 = max(etf_price-6200,0)
    part3 = -2*max(etf_price-6600,0)
    part4 = 3*max(etf_price-7000,0)
    return part1 + part2 + part3 + part4

In [15]:
class DummyBot(BaseBot):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # 历史数据/指标参数/交易参数/阈值参数定义与初始化
        self.symbols = ["TIDE_SPOT","TIDE_SWING","WX_SPOT","WX_SUM","LHR_COUNT","LHR_INDEX","LON_ETF","LON_FLY"]
        self.position_limit = 100

        self.price_history = {symbol:[] for symbol in self.symbols}
        self.feature_history = {symbol:[] for symbol in self.symbols}
        self.history_window = 20

        self.momentum_window = 5
        self.volatility_window = 10
        self.ma_short_window = 5
        self.ma_long_window = 15

        self.mm_base_qty = 5
        self.dir_medium_qty = 10
        self.dir_high_qty = 15

        self.high_confidence_threshold = 0.7
        self.medium_confidence_threshold = 0.4
        self.order_imbalance_threshold = 0.2
        
        self.running = False
        self.interval = 5

    def on_orderbook(self,orderbook):
        pass  
        
    def on_trades(self,trade):
        pass

    def collect_market_data(self):
        market_data = {}
        positions = bot.get_positions()
        for symbol in self.symbols:
            ob = self.get_orderbook(symbol)
            if not ob:
                continue
            best_bid = max(ob.buy_orders,key=lambda x:x.price).price if ob.buy_orders else 0
            best_bid_qty = max(ob.buy_orders,key=lambda x:x.price).volume if ob.buy_orders else 0
            total_bid_volume = sum(o.volume for o in ob.buy_orders)
            best_ask = min(ob.sell_orders,key=lambda x:x.price).price if ob.sell_orders else 0
            best_ask_qty = min(ob.sell_orders, key=lambda x:x.price).volume if ob.sell_orders else 0
            total_ask_volume = sum(o.volume for o in ob.sell_orders)
            mid_price = (best_bid+best_ask)/2 if (best_bid and best_ask) else get_settlement_price(symbol)
            
            current_position = positions.get(symbol,0)
            available_buy_qty = self.position_limit - current_position
            available_sell_qty = current_position + self.position_limit
            
            current_market_data = {
                'best_bid': best_bid,
                'best_ask': best_ask,
                'best_bid_qty': best_bid_qty,
                'best_ask_qty': best_ask_qty,
                'mid_price': mid_price,
                'total_bid_volume': total_bid_volume,
                'total_ask_volume': total_ask_volume,
                'current_position': current_position,
                'available_buy_qty': available_buy_qty,
                'available_sell_qty': available_sell_qty
            }
            market_data[symbol] = current_market_data
            self.update_history(symbol, mid_price, current_market_data)
        return market_data

    def update_history(self,symbol,mid_price,market_data):
        self.price_history[symbol].append(mid_price)
        if len(self.price_history[symbol]) > self.history_window:
            self.price_history[symbol].pop(0)
        self.feature_history[symbol].append(market_data)
        if len(self.feature_history[symbol]) > self.history_window:
            self.feature_history[symbol].pop(0)

    def calculate_features(self,market_data):
        features_dict = {}
        for symbol in self.symbols:
            current_data = market_data[symbol]
            price_history = self.price_history[symbol]
            features = {}

            # 基础价格特征/动量指标/移动平均指标/波动率指标/订单簿特征
            features['mid_price'] = current_data['mid_price']
            
            if len(price_history) >= self.momentum_window:
                short_momentum = (price_history[-1]-price_history[-self.momentum_window])/self.momentum_window
                features['momentum_short'] = short_momentum
            else:
                features['momentum_short'] = 0
            
            if len(price_history) >= self.ma_long_window:
                ma_short = np.mean(price_history[-self.ma_short_window:])
                ma_long = np.mean(price_history[-self.ma_long_window:])
                features['ma_short'] = ma_short
                features['ma_long'] = ma_long
                features['ma_crossover'] = ma_short-ma_long
            else:
                features['ma_short'] = 0
                features['ma_long'] = 0
                features['ma_crossover'] = 0

            if len(price_history) >= self.volatility_window:
                volatility = np.std(price_history[-self.volatility_window:])
                features['volatility'] = volatility
                features['volatility_ratio'] = volatility/np.mean(price_history[-self.volatility_window:]) if np.mean(price_history[-self.volatility_window:]) > 0 else 0
            else:
                features['volatility'] = 0
                features['volatility_ratio'] = 0
            
            total_volume = current_data['total_bid_volume']+current_data['total_ask_volume']
            if total_volume > 0:
                order_imbalance = (current_data['total_bid_volume']-current_data['total_ask_volume'])/total_volume
                features['order_imbalance'] = order_imbalance
            else:
                features['order_imbalance'] = 0
            
            features['depth_ratio'] = current_data['best_bid_qty']/current_data['best_ask_qty'] if current_data['best_ask_qty'] else 0
            
            features['market_active'] = 1 if (current_data['best_bid_qty'] > 0 and current_data['best_ask_qty'] > 0) else 0
            
            features['current_position'] = current_data['current_position']
            features['position_ratio'] = features['current_position']/self.position_limit
            
            features_dict[symbol] = features
        return features_dict

    def predict_price_direction(self,features):
        predictions = {}
        for symbol in self.symbols:
            feat = features[symbol]
            signal_scores = []
            
             # 动量信号(权重0.3)
            if abs(feat['momentum_short']) > 0.001:
                momentum_score = min(abs(feat['momentum_short'])*100,1)
                if feat['momentum_short'] > 0:
                    signal_scores.append(('buy',0.3*momentum_score))
                else:
                    signal_scores.append(('sell',0.3*momentum_score))
            
            # 移动平均交叉信号(权重0.25)
            if abs(feat['ma_crossover']) > 0.001:
                crossover_strength = min(abs(feat['ma_crossover'])*100,1)
                if feat['ma_crossover'] > 0:
                    signal_scores.append(('buy',0.25*crossover_strength))
                else:
                    signal_scores.append(('sell',0.25*crossover_strength)) 
            
            # 订单不平衡信号(权重0.25)
            if abs(feat['order_imbalance']) > self.order_imbalance_threshold:
                imbalance_strength = min(abs(feat['order_imbalance'])/0.5,1)
                if feat['order_imbalance'] > 0:
                    signal_scores.append(('buy',0.25*imbalance_strength))
                else:
                    signal_scores.append(('sell',0.25*imbalance_strength))
            
            # 买卖深度比信号(权重0.2)
            if feat['depth_ratio'] > 1.5:
                depth_strength = min((feat['depth_ratio']-1)/2,1)
                signal_scores.append(('buy',0.2*depth_strength))
            elif feat['depth_ratio'] < 2/3:  
                depth_strength = min((1/feat['depth_ratio']-1)/2,1) if feat['depth_ratio'] != 0 else 0
                signal_scores.append(('sell',0.2*depth_strength))
            
            # 汇总信号
            buy_score = sum(score for dir_, score in signal_scores if dir_ == 'buy')
            sell_score = sum(score for dir_, score in signal_scores if dir_ == 'sell')
            net_score = buy_score - sell_score
            total_strength = buy_score + sell_score
                
            # 计算置信度
            if total_strength < 0.3:
                clarity = 0.3
            elif total_strength > 0.7:
                clarity = 1.0
            else:
                clarity = 0.65
            total_confidence = abs(net_score)*clarity
        
            # 确定方向
            if net_score > 0.1:
                direction = 'buy'
                confidence = min(total_confidence,1.0)
            elif net_score < -0.1:
                direction = 'sell'
                confidence = min(total_confidence,1.0)
            else:
                direction = 'hold'
                confidence = 0

            # 市场状态调整置信度
            if not feat['market_active']:
                confidence *= 0.5
            if feat['volatility_ratio'] > 0.01:
                confidence *= 0.7
            
            predictions[symbol] = (direction,confidence)
        return predictions

    def make_market_making_decisions(self,market_data,features):
        mm_decisions = {}
        for symbol in self.symbols:
            data = market_data[symbol]
            feat = features[symbol]
            decision = {'bid':None,'ask':None}
            
            # 基础价格和挂单数量
            bid_price = data['best_bid']
            ask_price = data['best_ask']
            base_qty = self.mm_base_qty
            if bid_price == 0 or ask_price == 0:
                mm_decisions[symbol] = decision
                continue
            
            # 根据波动率调整数量
            volatility_ratio = feat['volatility_ratio']
            if volatility_ratio > 0.01:
                volatility_factor = 0.5
            elif volatility_ratio > 0.005:
                volatility_factor = 0.7
            else:
                volatility_factor = 1.0

            # 仓位调整
            position_ratio = abs(feat['position_ratio'])
            if position_ratio > 0.7:
                if feat['current_position'] > 0:
                    position_factor_bid = 0.3  
                    position_factor_ask = 1.5
                else:
                    position_factor_bid = 1.5
                    position_factor_ask = 0.3
            elif position_ratio > 0.4:
                if feat['current_position'] > 0:
                    position_factor_bid = 0.7
                    position_factor_ask = 1.3
                else:
                    position_factor_bid = 1.3
                    position_factor_ask = 0.7
            else:
                position_factor_bid = 1.0
                position_factor_ask = 1.0
            
            # 最终数量
            bid_qty = max(1,min(int(base_qty*volatility_factor*position_factor_bid),data['available_buy_qty']))
            ask_qty = max(1,min(int(base_qty*volatility_factor*position_factor_ask), data['available_sell_qty']))
            
            # 封装做市单
            if data['available_buy_qty'] >= 1:
                decision['bid'] = (bid_price,bid_qty)
            if data['available_sell_qty'] >= 1:
                decision['ask'] = (ask_price,ask_qty)
            
            mm_decisions[symbol] = decision
        return mm_decisions

    def make_directional_decisions(self,market_data,features,predictions):
        dir_decisions = {}
        for symbol in self.symbols:
            data = market_data[symbol]
            feat = features[symbol]
            direction,confidence = predictions[symbol]
            decision = {'bid': None,'ask': None}
            
            # 低置信度跳过
            if direction == 'hold' or confidence < self.medium_confidence_threshold:
                dir_decisions[symbol] = decision
                continue
            
            # 确定订单数量
            if confidence >= self.high_confidence_threshold:
                dir_qty = self.dir_high_qty
            else:
                dir_qty = self.dir_medium_qty
            
            # 波动率调整
            if feat['volatility_ratio'] > 0.01:
                dir_qty = int(dir_qty * 0.6)
            elif feat['volatility_ratio'] > 0.005:
                dir_qty = int(dir_qty * 0.8)
            
            # 确定价格和方向
            best_ask = data['best_ask']
            best_bid = data['best_bid']
            if direction == 'buy' and best_ask > 0:
                target_price = best_ask+1
                actual_qty = min(dir_qty,data['available_buy_qty'])
                decision['bid'] = (target_price,actual_qty)
            
            elif direction == 'sell' and best_bid > 0:
                target_price = best_bid-1 
                actual_qty = min(dir_qty,data['available_sell_qty'])
                decision['ask'] = (target_price, actual_qty)
            
            dir_decisions[symbol] = decision
        return dir_decisions

    def combine_orders(self,market_data,mm_decisions,dir_decisions):
        orders = []
        for symbol in self.symbols:
            data = market_data[symbol]
            mm_decision = mm_decisions[symbol]
            dir_decision = dir_decisions[symbol]
            final_bid_order = None
            final_ask_order = None
            
            # 合并做市和方向性买单, 使用更激进价格
            mm_bid = mm_decision['bid']
            dir_bid = dir_decision['bid']
            if mm_bid and dir_bid:
                mm_price, mm_qty = mm_bid
                dir_price, dir_qty = dir_bid
                final_price = max(mm_price,dir_price)
                final_qty = min(mm_qty+dir_qty, data['available_buy_qty'])
                if final_qty >= 1:
                    final_bid_order = (symbol,final_price,Side.BUY,final_qty)
            elif mm_bid:
                price, qty = mm_bid
                final_bid_order = OrderRequest(symbol,price,Side.BUY,qty)
            elif dir_bid:
                price, qty = dir_bid
                final_bid_order = OrderRequest(symbol,price,Side.BUY,qty)

            # 合并做市和方向性卖单, 使用更激进价格
            mm_ask = mm_decision['ask']
            dir_ask = dir_decision['ask']
            if mm_ask and dir_ask:
                mm_price, mm_qty = mm_ask
                dir_price, dir_qty = dir_ask
                final_price = min(mm_price,dir_price)
                final_qty = min(mm_qty+dir_qty, data['available_sell_qty'])
                if final_qty >= 1:
                    final_ask_order = OrderRequest(symbol,final_price,Side.SELL,final_qty)
            elif mm_ask:
                price, qty = mm_ask
                final_ask_order = OrderRequest(symbol,price,Side.SELL,qty)
            elif dir_ask:
                price, qty = dir_ask
                final_ask_order = OrderRequest(symbol,price,Side.SELL,qty)
            
            # 添加到订单列表
            if final_bid_order:
                orders.append(final_bid_order)
            if final_ask_order:
                orders.append(final_ask_order)
        return orders

    def run_bot(self, interval=5):
        try:
            while True:
                market_data = self.collect_market_data()
                features = self.calculate_features(market_data)
                predictions = self.predict_price_direction(features)
                mm_decisions = self.make_market_making_decisions(market_data,features)
                dir_decisions = self.make_directional_decisions(market_data,features,predictions)
                orders = self.combine_orders(market_data,mm_decisions,dir_decisions)
                for order in orders:
                    self.send_order(order)
                time.sleep(max(interval, 1))
                self.cancel_all_orders()
        
        except KeyboardInterrupt:
            # 取消所有订单，停止Bot
            self.cancel_all_orders()
            self.stop()
            print("Bot stopped gracefully")
        except Exception as e:
            print(f"Error running bot: {e}")
            self.cancel_all_orders()
            self.stop()

        

In [ ]:
bot = DummyBot(EXCHANGE_URL, USERNAME, PASSWORD)
bot.run_bot()